# 공시 변화 기반 재무위험 조기경보 — ML 노트북

2026 D&A ML Competition 가점 제출용. 가점 4항목(수업 모델 · 선택 근거 · 검증 · 실험/해석)을 장별로 대응합니다.

> ⚠️ 지금은 `USE_SYNTHETIC = True`로 **합성 데이터**를 씁니다. 실데이터(`data/processed/panel.csv`)를 만든 뒤 `False`로 바꾸고 처음부터 다시 실행하세요. 합성 데이터 결과는 제출·보고에 쓰지 않습니다.

## 0. 문제와 라벨 정의

- **예측 대상**: t년 사업보고서 기준 정보로, t+1년에 위험 사건이 일어날지
- **위험 사건(label=1)**: t+1년 ① 상장폐지 ② 관리종목 지정 ③ 감사의견 비적정(한정·부적정·의견거절) 중 하나
- **단위**: 기업 × 사업연도

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay
from dart_agent import model as M
from dart_agent.synthetic import make_panel

USE_SYNTHETIC = True
TEST_START = 2022   # 이 연도부터 test
plt.rcParams["axes.unicode_minus"] = False

## 1. 데이터 출처

| 데이터 | 제공처 | 접근 | 라이선스 | 수집일 |
|---|---|---|---|---|
| 재무제표, 감사의견, 공시 목록·원문 | 금융감독원 OpenDART | OpenDART API | OpenDART 이용약관 | (기입) |
| 상장폐지·관리종목 목록 | 한국거래소 KIND | 웹 다운로드 | KRX 이용약관 | (기입) |

전처리 상세는 `docs/data_sources.md`.

In [ ]:
df = make_panel() if USE_SYNTHETIC else pd.read_csv(ROOT / "data/processed/panel.csv", dtype={"corp_code": str})
print(df.shape)
print("연도별 위험 비율"); print(df.groupby("year")["label"].agg(["count", "mean"]).round(3))

## 2. 검증 설계 — 시간 기준 분할

무작위로 섞어 나누면 미래 연도의 정보가 학습에 들어가 성능이 부풀려집니다(데이터 누수). 그래서 **연도 기준**으로 과거로 학습하고 미래로 평가합니다. 또 변수는 t년 공시에서만 만들고, 다음 해 감사의견은 라벨에만 씁니다 (`scripts/build_dataset.py`의 `attach_labels`).

In [ ]:
FEATS_FIN = [f for f in M.FINANCIAL_FEATURES if f in df.columns]
FEATS_ALL = FEATS_FIN + [f for f in M.TEXT_FEATURES if f in df.columns]

train, test = M.time_split(df, TEST_START)
print(f"train {train.year.min()}~{train.year.max()} ({len(train)}행, 위험 {train.label.mean():.1%})")
print(f"test  {test.year.min()}~{test.year.max()} ({len(test)}행, 위험 {test.label.mean():.1%})")
assert train.year.max() < test.year.min()

## 3. 모델 선택 근거

- **Logistic Regression (기본)**: 금융 업무에서는 "왜 위험한가"를 설명해야 합니다. 계수의 부호와 크기로 각 변수가 위험을 올리는지 내리는지 바로 읽을 수 있어, Agent 리포트의 근거로 씁니다.
- **Decision Tree (기본)**: "자본잠식률 > x 이고 영업현금흐름 적자면 위험" 같은 규칙으로 판단 과정을 보여줄 수 있습니다. 깊이를 4로 제한해 해석 가능성을 지킵니다.
- **Random Forest (비교용)**: 변수 간 비선형 관계를 잡아 성능 상한을 가늠합니다. 설명력은 떨어지므로 성능 비교용으로만 씁니다.
- **불균형 대응**: 위험 기업이 소수라 세 모델 모두 `class_weight="balanced"`로 소수 클래스에 가중치를 줍니다.

## 4. 학습과 5. 평가 지표

위험 기업 비율이 약 5%라, 모두 "안전"으로 찍어도 Accuracy는 95%가 됩니다. 그래서 Accuracy 대신
- **Recall**: 실제 위험 기업 중 놓치지 않은 비율 (조기경보에서 가장 중요)
- **PR-AUC**: 임계값과 무관하게 소수 클래스 탐지 성능. 무작위 기준선 = 위험 비율

을 주 지표로 씁니다.

In [ ]:
models = M.build_models()
res = M.evaluate(models, train[FEATS_ALL], train.label, test[FEATS_ALL], test.label)
res.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for name, m in models.items():
    PrecisionRecallDisplay.from_estimator(m, test[FEATS_ALL], test.label, name=name, ax=ax)
ax.axhline(test.label.mean(), ls="--", c="gray", label="random baseline")
ax.legend(); ax.set_title("Precision-Recall (test)")
plt.show()

## 6. 해석 — 변수 중요도

In [ ]:
imp = pd.DataFrame({
    "Logistic(표준화 계수)": M.feature_importance(models["Logistic"], FEATS_ALL),
    "RandomForest(중요도)": M.feature_importance(models["RandomForest"], FEATS_ALL),
}).sort_values("RandomForest(중요도)", ascending=False)
imp.round(3)

In [ ]:
from sklearn.tree import export_text
print(export_text(models["DecisionTree"].named_steps["clf"], feature_names=FEATS_ALL, max_depth=3))

## 7. 실험 — 문구 변화 변수를 넣으면 성능이 오르는가

재무 변수만 쓴 모델과, 공시 문구 변화 변수(감사보고서·주석 TF-IDF 변화량, 새 경고 문구 수)를 더한 모델을 같은 분할에서 비교합니다.

In [ ]:
fin_only = M.evaluate(M.build_models(), train[FEATS_FIN], train.label, test[FEATS_FIN], test.label)
compare = pd.concat({"재무만": fin_only[["recall", "pr_auc"]], "재무+문구": res[["recall", "pr_auc"]]}, axis=1)
compare[("차이", "pr_auc")] = compare[("재무+문구", "pr_auc")] - compare[("재무만", "pr_auc")]
compare.round(3)

## 8. 결론 (실데이터로 다시 작성)

- 주 모델: (기입)
- 문구 변화 변수의 기여: (기입)
- 한계: 위험 사건 수가 적음, 문구 섹션 추출이 단순함, 업종 차이 미반영